# 📖 Notebook 3: Real-Time Collaboration via WebSockets

Now that we understand OT and CRDTs, let's build the **real-time experience**. In this notebook, we connect to our live doc server over WebSockets, send edits, and see how multiple users collaborate on the same document.

## Learning Objectives

By the end of this notebook, you'll understand:
- Why WebSockets are used for collaborative editing (not HTTP)
- The connection lifecycle: connect → join document → edit → disconnect
- How cursor presence works (seeing where other users are typing)
- The server-side broadcast pattern for real-time sync

## 🛠️ Setup

Make sure the doc server is running:

```bash
cd 06-system-designs/google-docs
docker compose up -d
```

Verify the server is up:
```bash
docker logs googledocs-doc-server
```

You should see: `🚀 Google Docs Collaboration Server starting on ws://0.0.0.0:8765`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import asyncio
import json
import websockets
import redis

WS_URL = "ws://localhost:8765"
REDIS_CONFIG = {"host": "localhost", "port": 6379, "decode_responses": True}

def get_redis():
    return redis.Redis(**REDIS_CONFIG)

# Test connections
async def test_ws():
    try:
        async with websockets.connect(WS_URL) as ws:
            await ws.send(json.dumps({"type": "connect", "user_id": 1}))
            resp = json.loads(await ws.recv())
            assert resp["type"] == "connected"
            print("✅ WebSocket server is running")
    except Exception as e:
        print(f"❌ WebSocket failed: {e}")
        print("   Run: docker compose up -d")

try:
    r = get_redis()
    r.ping()
    print("✅ Redis connected")
except Exception as e:
    print(f"❌ Redis failed: {e}")

await test_ws()

## 🤔 Why WebSockets?

In a collaborative editor, the server needs to **push** changes to clients the moment another user types. Regular HTTP can't do this efficiently:

| Approach | How It Works | Problem |
|----------|-------------|----------|
| **HTTP Polling** | Client asks "any changes?" every 100ms | Wasteful — most polls return nothing |
| **Long Polling** | Client waits for a response until something changes | Reconnection overhead, not truly real-time |
| **Server-Sent Events** | Server pushes events to client | One-directional only (server → client) |
| **WebSockets** ✅ | Persistent bi-directional connection | Both sides can send messages anytime |

WebSockets are the standard choice for collaborative editors because:
- **Bi-directional**: client sends edits, server broadcasts to others
- **Low latency**: no connection setup overhead per message
- **Persistent**: stays open for the entire editing session

In [ ]:
# Let's compare: How many bytes does it take to send an edit?

http_body = json.dumps({"op_type": "insert", "position": 5, "content": "x"})

# A realistic HTTP POST: request line + headers + blank line + body.
http_headers = (
    "POST /api/docs/1/edit HTTP/1.1\r\n"
    "Host: localhost:8000\r\n"
    "Content-Type: application/json\r\n"
    "Authorization: Bearer token123\r\n"
    f"Content-Length: {len(http_body)}\r\n"
    "\r\n"
)
http_total = len(http_headers) + len(http_body)

# WebSocket message (after the connection is already established)
ws_message = json.dumps({"type": "edit", "document_id": 1, "op_type": "insert", "position": 5, "content": "x"})
ws_total = len(ws_message) + 6  # ~6 bytes WebSocket frame overhead (masked client frame)

print("📊 Bytes Per Edit")
print("=" * 40)
print(f"HTTP POST:  {http_total:>4} bytes  (headers + body)")
print(f"WebSocket:  {ws_total:>4} bytes  (frame + payload)")
print(f"Savings:    {((http_total - ws_total) / http_total * 100):.0f}%")
print()
print("At 5 keystrokes/second for 10 users, that's:")
print(f"  HTTP:      {http_total * 5 * 10:,} bytes/sec")
print(f"  WebSocket: {ws_total * 5 * 10:,} bytes/sec")
print()
print("💡 WebSockets avoid resending headers on every message.")
print("   (The HTTP number is optimistic: no cookies, no User-Agent, no TLS")
print("    handshake, and no response headers coming back.)")

assert ws_total < http_total, "the comparison stopped favouring WebSockets — check the numbers"

## 🔌 The Connection Lifecycle

Here's what happens when a user opens a document:

```
1. CONNECT        →  Authenticate with user_id
2. JOIN_DOC       →  Join a specific document for editing
3. RECEIVE STATE  ←  Server sends current document text + who's editing
4. EDIT LOOP      ↔  Send edits, receive others' edits (continuous)
5. DISCONNECT     →  Connection closes, server removes from presence
```

Let's walk through each step.

In [ ]:
# Helper class to make WebSocket interactions easier in notebooks

class DocClient:
    """A simple Google Docs client for notebook demos.

    Two things make this more than a `send()` wrapper:

    * it tracks a **revision** -- how many server operations it has seen. The
      server needs that number to know which concurrent ops to transform an
      incoming edit against.
    * it applies whatever the server says it applied, not what the client
      *hoped* it applied. A real editor updates the screen optimistically and
      reconciles afterwards; waiting for the ACK is the simple version of the
      same idea and keeps this notebook honest about who is authoritative.
    """

    def __init__(self, user_id, name):
        self.user_id = user_id
        self.name = name
        self.ws = None
        self.doc_text = ""
        self.doc_id = None
        self.revision = 0     # how many server ops this client has seen
        self.cursor = 0
        self.messages = []    # messages that arrived while we waited for another

    async def connect(self):
        """Step 1: Open WebSocket and authenticate."""
        self.ws = await websockets.connect(WS_URL)
        await self.ws.send(json.dumps({"type": "connect", "user_id": self.user_id}))
        resp = json.loads(await self.ws.recv())
        print(f"  [{self.name}] Connected: {resp}")

    async def join_doc(self, doc_id):
        """Step 2: Join a document for editing."""
        self.doc_id = doc_id
        await self.ws.send(json.dumps({"type": "join_doc", "document_id": doc_id}))

        state = await self.wait_for("doc_state")
        assert state is not None, f"{self.name} never received doc_state for doc {doc_id}"
        assert "revision" in state, (
            "this doc_state has no 'revision' field, so the server predates the OT "
            "rework and will never transform anything. Rebuild the image:\n"
            "    docker compose up -d --build"
        )
        print(f"  [{self.name}] Joined doc {doc_id} (version {state['version']}, revision {self.revision})")
        preview = self.doc_text if len(self.doc_text) <= 80 else self.doc_text[:80] + "..."
        print(f"  [{self.name}] Document: '{preview}'")

        presence = await self.wait_for("presence_list")
        users = presence.get("users", []) if presence else []
        if users:
            print(f"  [{self.name}] Other editors: {', '.join(u['name'] for u in users)}")
        else:
            print(f"  [{self.name}] No other editors currently")

    # -- local document maintenance ---------------------------------------
    def _apply(self, op):
        """Apply one operation (from an ACK or a remote_op) to our local text."""
        pos = op["position"]
        if op["op_type"] == "insert":
            self.doc_text = self.doc_text[:pos] + op.get("content", "") + self.doc_text[pos:]
        else:
            self.doc_text = self.doc_text[:pos] + self.doc_text[pos + op.get("length", 1):]

    # -- sending edits -----------------------------------------------------
    async def send_edit(self, op_type, position, content="", length=0, revision=None, wait=True):
        """Send one edit.

        `revision` defaults to whatever we have seen. Passing an older revision
        on purpose is how the demos below make two edits genuinely concurrent.
        `wait=False` returns before the ACK, so a second client can get its edit
        in first.
        """
        await self.ws.send(json.dumps({
            "type": "edit",
            "document_id": self.doc_id,
            "op_type": op_type,
            "position": position,
            "content": content,
            "length": length,
            "revision": self.revision if revision is None else revision,
        }))
        return await self.collect_ack() if wait else None

    async def collect_ack(self, timeout=2.0):
        """Wait for our ACK and apply the op(s) the server actually applied."""
        ack = await self.wait_for("ack", timeout=timeout)
        assert ack is not None, f"{self.name} never got an ACK for its edit"
        assert "transformed" in ack, (
            "the server ACKed without saying which op it actually applied, so this "
            "client cannot stay in sync. Rebuild the image:\n"
            "    docker compose up -d --build"
        )
        for op in ack["transformed"]:
            self._apply(op)
        self.revision = ack.get("revision", self.revision)
        return ack

    async def insert(self, position, content, revision=None):
        """Send an insert operation."""
        ack = await self.send_edit("insert", position, content=content, revision=revision)
        print(f"  [{self.name}] INSERT({position}, '{content}') → ACK (revision {self.revision})")
        return ack

    async def delete(self, position, length=1, revision=None):
        """Send a delete operation."""
        ack = await self.send_edit("delete", position, length=length, revision=revision)
        print(f"  [{self.name}] DELETE({position}, {length}) → ACK (revision {self.revision})")
        return ack

    async def update_cursor(self, position):
        """Send a cursor position update."""
        self.cursor = position
        await self.ws.send(json.dumps({
            "type": "cursor_update",
            "document_id": self.doc_id,
            "position": position,
        }))

    # -- receiving ---------------------------------------------------------
    async def recv_message(self, timeout=0.5):
        """Receive one message (with timeout), keeping our local copy in step."""
        try:
            raw = await asyncio.wait_for(self.ws.recv(), timeout=timeout)
        except asyncio.TimeoutError:
            return None
        msg = json.loads(raw)
        if msg["type"] == "remote_op":
            self._apply(msg)
            self.revision = msg.get("revision", self.revision)
        elif msg["type"] == "doc_state":
            self.doc_text = msg["text"]
            self.revision = msg.get("revision", 0)
        self.messages.append(msg)
        return msg

    async def wait_for(self, msg_type, timeout=2.0):
        """Read until a message of `msg_type` shows up.

        Messages that arrive first are not dropped -- they are applied on the
        way past and buffered, which is exactly what a real client's receive
        loop does. Reading blindly with recv() instead is how notebook demos
        end up asserting on whichever message happened to win the race.
        """
        for i, m in enumerate(self.messages):
            if m["type"] == msg_type:
                return self.messages.pop(i)
        loop = asyncio.get_running_loop()
        deadline = loop.time() + timeout
        while True:
            remaining = deadline - loop.time()
            if remaining <= 0:
                return None
            msg = await self.recv_message(timeout=remaining)
            if msg is None:
                return None
            if msg["type"] == msg_type:
                self.messages.pop()   # it was just appended by recv_message
                return msg

    async def recv_all(self, timeout=0.5):
        """Receive all pending messages."""
        messages = []
        while True:
            msg = await self.recv_message(timeout)
            if msg is None:
                break
            messages.append(msg)
        return messages

    async def disconnect(self):
        """Close the WebSocket connection."""
        if self.ws:
            await self.ws.close()
            print(f"  [{self.name}] Disconnected")

print("DocClient helper class defined! ✅")

In [ ]:
# Step 1 & 2: Connect and join a document

alice = DocClient(user_id=1, name="Alice")
await alice.connect()
await alice.join_doc(1)  # Join "Meeting Notes — Q1 Planning"

In [ ]:
# Step 3: Bob joins the same document — Alice gets a notification!

bob = DocClient(user_id=2, name="Bob")
await bob.connect()
await bob.join_doc(1)  # Join the same document

# Alice should receive a "user_joined" notification
alice_msg = await alice.wait_for("user_joined", timeout=2)
assert alice_msg is not None, "Alice was never told that Bob joined"
print(f"\n  Alice received: {alice_msg['type']} → {alice_msg.get('name', '')} joined!")

# Both clients loaded the same document at the same revision.
assert alice.doc_text == bob.doc_text, "the two clients disagree about the document"
assert alice.revision == bob.revision

In [ ]:
# Step 4: Alice makes an edit — Bob receives it in real-time!

print("Alice types at the end of the document...")
insert_pos = len(alice.doc_text)
await alice.insert(insert_pos, "\n- ACTION: Schedule follow-up meeting")

# Bob should receive the remote operation (DocClient applies it as it arrives)
bob_msg = await bob.wait_for("remote_op", timeout=2)
assert bob_msg is not None, "Bob never received Alice's edit"
print(f"\n  Bob received remote_op: {bob_msg['op_type'].upper()} at position {bob_msg['position']}")
print(f"  Content: '{bob_msg.get('content', '')[:50]}'")
print(f"  Bob's document now ends with: '...{bob.doc_text[-50:]}'")

assert alice.doc_text == bob.doc_text, "clients diverged after a single sequential edit"

In [ ]:
# Step 4 continued: Bob edits too — Alice receives it!

print("Bob types at the end of the document...")
insert_pos = len(bob.doc_text)
await bob.insert(insert_pos, "\n- ACTION: Review budget proposal")

# Alice should receive it
alice_msg = await alice.wait_for("remote_op", timeout=2)
assert alice_msg is not None, "Alice never received Bob's edit"
print(f"\n  Alice received Bob's edit")
print(f"  Both documents now match: {alice.doc_text == bob.doc_text}")

assert alice.doc_text == bob.doc_text, (
    f"clients diverged:\n  alice: ...{alice.doc_text[-60:]!r}\n  bob:   ...{bob.doc_text[-60:]!r}"
)
assert alice.revision == bob.revision, "clients are at different revisions"

## ⚡ Actually Concurrent: Two Clients, One Revision

Everything above was **sequential**. Alice waited for her ACK before Bob typed,
so the server never had to transform anything — and a lab that only does this
cannot tell you whether its OT works at all.

Real editors do not take turns. To reproduce that, both clients send an edit
quoting the **same revision number**: each one believes the document is exactly
as it was when they last looked. The server sees the second edit arrive against
a document that has already moved underneath it, and has to transform it.

The edits are chosen so the failure is unmistakable:

- Alice inserts a label at position 0
- Bob deletes 7 characters at position 0 — the scratch word we plant first

Untransformed, Bob's delete lands on **Alice's label** instead of the word he
selected. Transformed, it shifts right by the length of the label and removes
what Bob actually meant.

In [ ]:
# Plant a scratch word so the concurrent edits are self-cleaning.
SCRATCH = "SCRATCH"
await alice.insert(0, SCRATCH)
await bob.wait_for("remote_op", timeout=2)

start_rev = alice.revision
assert bob.revision == start_rev, f"clients out of step: {alice.revision} vs {bob.revision}"
baseline = alice.doc_text
print(f"Both clients at revision {start_rev}: '{baseline[:40]}...'")
print()

# Now the concurrent part: fire both edits before either ACK comes back, and
# make both of them quote `start_rev`.
LABEL = "PREFIX: "
await alice.send_edit("insert", 0, content=LABEL, revision=start_rev, wait=False)
await bob.send_edit("delete", 0, length=len(SCRATCH), revision=start_rev, wait=False)

alice_ack = await alice.collect_ack()
bob_ack = await bob.collect_ack()

print(f"Alice sent INSERT(0, '{LABEL}')  → server applied {alice_ack['transformed']}")
print(f"Bob   sent DELETE(0, {len(SCRATCH)})        → server applied {bob_ack['transformed']}")
print()

# Drain the cross-broadcasts so both local copies are complete.
await alice.wait_for("remote_op", timeout=2)
await bob.wait_for("remote_op", timeout=2)

print(f"Alice: '{alice.doc_text[:40]}...'")
print(f"Bob:   '{bob.doc_text[:40]}...'")
print()

# What the server WOULD have produced with no transformation at all: apply both
# ops blindly, in arrival order, to the document both clients started from.
naive = LABEL + baseline                         # Alice's insert
naive = naive[len(SCRATCH):]                     # Bob's delete, untransformed
print(f"Without OT the server would have stored: '{naive[:40]}...'")
print(f"  — Bob's Backspace would have eaten the front of Alice's label ❌")

assert alice.doc_text == bob.doc_text, (
    f"clients diverged:\n  alice: {alice.doc_text[:60]!r}\n  bob:   {bob.doc_text[:60]!r}"
)
assert alice.doc_text.startswith(LABEL), (
    f"the concurrent delete corrupted Alice's insert: {alice.doc_text[:40]!r}"
)
assert SCRATCH not in alice.doc_text, "Bob's delete did not remove the word he selected"
assert alice.doc_text == LABEL + baseline[len(SCRATCH):], "unexpected merge result"
assert not naive.startswith(LABEL), "the untransformed demo is no longer broken"

print()
print("✅ Both intents preserved. Note this is the SERVER transforming; a real")
print("   client also transforms incoming ops against its own un-ACKed edits, which")
print("   this DocClient sidesteps by applying only what the ACK confirms.")

# Clean up: remove the label so the document is left as we found it.
await alice.delete(0, len(LABEL))
await bob.wait_for("remote_op", timeout=2)
assert alice.doc_text == baseline[len(SCRATCH):], "the demo did not clean up after itself"
print(f"\n🧹 Document restored: '{alice.doc_text[:40]}...'")

## 👆 Cursor Presence

One of the most important UX features: seeing **where** other users are editing. This is done by broadcasting cursor positions through the server and storing them in Redis.

In [ ]:
# Alice moves her cursor to position 50
await alice.update_cursor(50)

# Bob should receive the cursor update
bob_msg = await bob.wait_for("cursor_update", timeout=2)
assert bob_msg is not None, "Bob never saw Alice's cursor move"
print(f"Bob sees Alice's cursor at position {bob_msg['position']}")

# Check Redis for presence data
r = get_redis()
presence = r.hgetall("doc:1:presence")
print(f"\n📍 Presence data in Redis for doc 1:")
for user_id, data in presence.items():
    info = json.loads(data)
    print(f"  User {user_id}: {info['name']} (cursor: {info['cursor']}, color: {info['color']})")

print(f"\n💡 Redis stores presence for cross-server awareness.")
print(f"   In a multi-server setup, server B can read server A's user presence.")

assert json.loads(presence[str(alice.user_id)])["cursor"] == 50

### Cursors Have to Be Transformed Too

A cursor is a position, and every position in the document is invalidated by the
same edits that invalidate an operation. If someone types a paragraph above your
caret and nobody moves your stored cursor, your presence dot slides backwards
through the text — you appear to be somewhere you are not, and if the client
uses that number to place your next keystroke, you type in the wrong place.

The server runs the same shift rules over every other editor's cursor whenever
it applies an operation.

In [ ]:
# Bob parks his caret in the middle of the document; Alice types above it.

await bob.update_cursor(40)
await alice.wait_for("cursor_update", timeout=2)   # Alice hears about Bob's caret

r = get_redis()
def stored_cursor(user_id):
    return json.loads(r.hgetall("doc:1:presence")[str(user_id)])["cursor"]

before = stored_cursor(bob.user_id)
char_at_caret = alice.doc_text[before]

typed = ">>> "
await alice.insert(0, typed)                        # Alice types at the very start
await bob.wait_for("remote_op", timeout=2)          # Bob's client sees it

after = stored_cursor(bob.user_id)
print(f"Bob's caret before Alice typed: {before}  (on character {char_at_caret!r})")
print(f"Alice inserted {len(typed)} characters at position 0")
print(f"Bob's caret after:              {after}  (on character {bob.doc_text[after]!r})")

assert after == before + len(typed), (
    f"cursor was not transformed: expected {before + len(typed)}, Redis says {after}"
)
assert bob.doc_text[after] == char_at_caret, "the caret is pointing at a different character"
print("\n✅ The caret followed its character instead of staying on a stale offset.")

# Undo the scratch insert so the document is left as we found it.
await alice.delete(0, len(typed))
await bob.wait_for("remote_op", timeout=2)
assert stored_cursor(bob.user_id) == before, "the caret did not follow the delete back"
assert alice.doc_text == bob.doc_text

In [ ]:
# Step 5: Bob disconnects — Alice gets notified

await bob.disconnect()

alice_msg = await alice.wait_for("user_left", timeout=2)
assert alice_msg is not None, "Alice was never told that Bob left"
print(f"Alice received: user_left (user_id={alice_msg['user_id']})")

# Check Redis — Bob should be removed
presence = r.hgetall("doc:1:presence")
print(f"\nPresence after Bob left:")
for user_id, data in presence.items():
    info = json.loads(data)
    print(f"  User {user_id}: {info['name']}")

assert str(bob.user_id) not in presence, "Bob's presence entry outlived his connection"
assert str(alice.user_id) in presence, "Alice is still editing but vanished from presence"

## 🔐 Role-Based Permissions

Not every collaborator should be able to edit. Google Docs has three roles:

| Role | Can view | Can edit | Can share |
|------|----------|----------|-----------|
| **Owner** | ✅ | ✅ | ✅ |
| **Editor** | ✅ | ✅ | ❌ |
| **Viewer** | ✅ | ❌ | ❌ |

The server must enforce this **server-side** — never trust the client.
In our seed data, Charlie (user 3) is a **viewer** on document 3. Let's
verify that when Charlie tries to edit, the server rejects the operation.

In [ ]:
# Charlie (user 3) is a VIEWER on document 3 — edits should be rejected

charlie = DocClient(user_id=3, name="Charlie")
await charlie.connect()
await charlie.join_doc(3)  # Shared Shopping List — Charlie's role is 'viewer'

before = charlie.doc_text

# Try to edit — server should respond with an error
await charlie.ws.send(json.dumps({
    "type": "edit",
    "document_id": 3,
    "op_type": "insert",
    "position": 0,
    "content": "evil edit",
    "revision": charlie.revision,
}))
resp = await charlie.wait_for("error", timeout=2)
print(f"Server response to Charlie's edit: {resp}")

assert resp is not None, "the server accepted an edit from a viewer (no error came back)"
print(f"\n✅ Viewer correctly blocked from editing.")

# ...and it must be a rejection, not just a complaint: no ACK, no document change.
assert await charlie.wait_for("ack", timeout=0.5) is None, "the viewer's edit was ACKed"
assert charlie.doc_text == before

await charlie.disconnect()

print(f"\n💡 Permissions are checked on EVERY edit on the server.")
print(f"   The client UI can disable the editor for viewers, but the server")
print(f"   is the true source of security — it never trusts client messages.")

## 🏗️ Scaling WebSocket Connections

With millions of users, one server can't handle all connections. The system design solution:

```
┌──────────┐     ┌──────────┐     ┌──────────┐
│ Client A  │     │ Client B  │     │ Client C  │
└─────┬────┘     └─────┬────┘     └─────┬────┘
      │               │               │
      ▼               ▼               ▼
┌───────────┐   ┌───────────┐   ┌───────────┐
│ Doc Server│   │ Doc Server│   │ Doc Server│
│    #1     │   │    #2     │   │    #3     │
└─────┬─────┘   └─────┬─────┘   └─────┬─────┘
      │               │               │
      └───────────────┼───────────────┘
                      │
             ┌────────▼────────┐
             │ Consistent Hash │
             │   Ring (ZK)     │
             └─────────────────┘
```

**Key points:**
- **Consistent hashing** routes all editors of the same document to the same server
- **OT requires** all editors on one server (central authority for operation ordering)
- When a server fails, its documents are redistributed to other servers
- Client reconnects to the correct server via hash ring lookup

In [ ]:
# Routing documents to servers: modulo hashing vs a real hash ring
import bisect
import hashlib


def h(key):
    """Hash a string to a point on a 128-bit circle."""
    return int(hashlib.md5(str(key).encode()).hexdigest(), 16)


def modulo_route(doc_key, servers):
    """The version everybody writes first: hash % number_of_servers."""
    return servers[h(doc_key) % len(servers)]


class HashRing:
    """Consistent hashing.

    Every server is placed at `vnodes` points around a circle. A document goes
    to the first server clockwise from its own point. Removing a server only
    reassigns the documents that were sitting on that server's arcs — which is
    the entire reason to prefer this over `hash % n`.
    """

    def __init__(self, servers, vnodes=100):
        self.vnodes = vnodes
        self.ring = []                 # sorted [(point, server), ...]
        for s in servers:
            self.add(s)

    def add(self, server):
        self.ring.extend((h(f"{server}#{i}"), server) for i in range(self.vnodes))
        self.ring.sort()

    def remove(self, server):
        self.ring = [(p, s) for p, s in self.ring if s != server]

    def route(self, doc_key):
        i = bisect.bisect_left(self.ring, (h(doc_key),))
        return self.ring[i % len(self.ring)][1]      # wrap around the circle


servers = [f"doc-server-{i}" for i in range(5)]
docs = [f"doc:{i}" for i in range(1, 1001)]
ring = HashRing(servers)

before_mod = {d: modulo_route(d, servers) for d in docs}
before_ring = {d: ring.route(d) for d in docs}

print(f"📊 Document → Server Routing ({len(servers)} servers, {len(docs)} documents)")
print("=" * 60)
for name in servers:
    n = sum(1 for s in before_ring.values() if s == name)
    print(f"  {name}: {'█' * (n // 10)} ({n} docs)")

# Now doc-server-2 crashes. Every document it was hosting HAS to move.
# Everything else should stay exactly where it is -- its editors are mid-session.
dead = "doc-server-2"
survivors = [s for s in servers if s != dead]
after_mod = {d: modulo_route(d, survivors) for d in docs}
ring.remove(dead)
after_ring = {d: ring.route(d) for d in docs}

must_move = sum(1 for d in docs if before_ring[d] == dead)
moved_mod = sum(1 for d in docs if before_mod[d] != after_mod[d])
moved_ring = sum(1 for d in docs if before_ring[d] != after_ring[d])

print()
print(f"💥 {dead} dies. Documents that MUST be reassigned: {must_move}")
print("-" * 60)
print(f"  hash % n:          {moved_mod:>4} documents move  ({moved_mod / len(docs):.0%})  ❌")
print(f"  consistent hash:   {moved_ring:>4} documents move  ({moved_ring / len(docs):.0%})  ✅")
print()
print("Every moved document is a disconnect: its editors have to reconnect and")
print("their in-memory OT state is rebuilt from the last snapshot. `hash % n`")
print("does that to almost every document in the fleet to lose one server.")

assert moved_ring == must_move, (
    f"the ring moved {moved_ring} documents but only {must_move} lived on {dead} "
    f"— that is not consistent hashing"
)
assert moved_mod > 3 * must_move, (
    f"modulo hashing only moved {moved_mod} documents; the demo is not showing "
    f"the problem consistent hashing exists to solve"
)

print()
print(f"💡 All editors of the same document still land on the same server —")
print(f"   critical for OT, since one server has to order every op for a document.")

In [ ]:
# Clean up connections
await alice.disconnect()

## 🧹 Cleanup

In [ ]:
# Clean up Redis presence keys
r = get_redis()
for key in r.keys("doc:*:presence"):
    r.delete(key)
print("🧹 Cleaned up Redis presence keys")

## 📚 Summary

### Key Takeaways

1. **WebSockets are essential** — bi-directional, low-latency, persistent connections for real-time editing
2. **Connection lifecycle**: connect → join doc → receive state → edit loop → disconnect
3. **Revisions make edits concurrent** — a client quotes the revision it edited against, and the server transforms anything that arrived since
4. **Cursor presence** uses Redis hashes — ephemeral, and transformed by every edit so a caret follows its character
5. **Broadcast pattern**: server transforms, applies, ACKs the sender with what it actually applied, then broadcasts to everyone else
6. **Scaling**: a consistent hash *ring* routes all editors of a document to one server, and moves only that server's documents when it dies

### What This Notebook's Client Does Not Do

- **No optimistic local echo.** `DocClient` waits for the ACK and applies what the
  server confirms. A real editor renders the keystroke immediately and keeps a
  queue of un-ACKed ops, transforming every incoming remote op against that queue.
- **No reconnect/resync.** If the socket drops mid-edit, our client has no way to
  replay from its last revision; real ones re-join and fast-forward.
- **No undo.** Undo in a collaborative editor is not "pop the last op" — it is
  "apply the inverse of *my* last op, transformed against everything that has
  happened since", which needs the same op history the server keeps.
- **Presence is single-server.** We write to Redis so a second server *could*
  read it, but nothing publishes cross-server op streams (that would be Redis
  pub/sub or a message bus per document).

### For System Design Interviews

- Always mention **WebSockets** for real-time collaborative features
- Discuss the **single-server-per-document** constraint from OT
- Know how **consistent hashing** distributes documents across servers, and why `hash % n` is not it
- Mention **Redis pub/sub or presence** for cross-server awareness

### Next Up

In **Notebook 4**, we'll explore **document versioning and history** — how snapshots, compaction, and version restore work.